In [1]:
import numpy as np
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from glob import glob
import librosa
import librosa.display
import IPython.display as ipd
from itertools import cycle
sns.set_theme(style="white", palette=None)
color_pal = plt.rcParams["axes.prop_cycle"].by_key()["color"]
color_cycle = cycle(plt.rcParams["axes.prop_cycle"].by_key()["color"])

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder,OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,classification_report
from sklearn.preprocessing import StandardScaler,PowerTransformer

In [3]:
audio_files=glob(r"C:\Users\abdul\OneDrive\Desktop\ineuronreso\Audio-Data-Project\ravdes-audio-project\RAVDESS-AUDIO-DATASET\**\*.wav", recursive=True)

In [4]:
print(f"Found {len(audio_files)} audio files.")

Found 923 audio files.


In [5]:
features = []
labels = []

In [6]:
for file in audio_files:
    try:
        y, sr = librosa.load(file)  # Load the audio file

        # --- MFCC ---
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
        mfccs_mean = np.mean(mfccs, axis=1)
        mfccs_std = np.std(mfccs, axis=1)

        # --- Chroma ---
        chroma = librosa.feature.chroma_stft(y=y, sr=sr)
        chroma_mean = np.mean(chroma, axis=1)
        chroma_std = np.std(chroma, axis=1)

        # --- Zero Crossing Rate ---
        zcr = librosa.feature.zero_crossing_rate(y)
        zcr_mean = np.mean(zcr)
        zcr_std = np.std(zcr)

        # --- RMS Energy ---
        rms = librosa.feature.rms(y=y)
        rms_mean = np.mean(rms)
        rms_std = np.std(rms)

         # --- Tonnetz ---
        tonnetz = librosa.feature.tonnetz(y=librosa.effects.harmonic(y), sr=sr)
        tonnetz_mean = np.mean(tonnetz, axis=1)
        tonnetz_std = np.std(tonnetz, axis=1)

         # --- Spectral Contrast ---
        contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
        contrast_mean = np.mean(contrast, axis=1)
        contrast_std = np.std(contrast, axis=1)


        # --- Combine all features ---
        feature_vector = np.concatenate([
            mfccs_mean, mfccs_std,
            chroma_mean, chroma_std,
            [zcr_mean,zcr_std],
            [rms_mean,rms_std],
            tonnetz_mean,tonnetz_std,
            contrast_mean,contrast_std
         ])

        # --- Correct label extraction ---
        filename = os.path.splitext(os.path.basename(file))[0]
        parts = filename.split("-")
        emotion_map = {
            '01': 'neutral',
            '02': 'calm',
            '03': 'happy',
            '04': 'sad',
            '05': 'angry',
            '06': 'fearful',
            '07': 'disgust',
            '08': 'surprised'
        }
        emotion_code = parts[2]
        label = emotion_map.get(emotion_code, "unknown")

        features.append(feature_vector)
        labels.append(label)

    except Exception as e:
        print(f"Failed to process {file}: {e}")


c:\Users\abdul\AppData\Local\Programs\Python\Python310\lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=1024 is too large for input signal of length=1012
  warnings.warn(


In [7]:
# Create a DataFrame for easier processing and visualization
df = pd.DataFrame(features)
df['labels'] = labels

In [8]:
df

,0,1,2,3,4,5,6,7,8,9,...,71,72,73,74,75,76,77,78,79,labels
0,-697.792603,54.890041,0.663465,12.435786,7.733952,0.530750,-3.216631,-3.159394,-10.977551,-2.848711,...,16.763104,45.346246,10.712655,4.592027,5.519944,3.719688,3.160476,4.936189,4.376869,neutral
1,-692.855774,55.363895,-1.548319,16.038305,8.818810,-0.146586,-1.373392,-5.293180,-11.623182,-1.348284,...,16.843068,44.918227,10.221810,4.672584,5.678050,4.050266,3.132121,4.980014,4.449650,neutral
2,-691.587891,58.024662,0.159465,13.624650,5.374113,1.162337,-2.083360,-5.382585,-10.332824,-3.662081,...,16.694574,45.031481,10.610992,5.361405,6.009305,3.831685,2.924071,4.630189,4.303702,neutral
3,-685.105469,55.879421,2.783262,13.252023,6.989670,2.981274,-1.586029,-6.961661,-10.348489,-3.270769,...,16.067050,44.119110,9.945630,4.248116,4.827358,3.778959,2.933074,3.786002,4.105325,neutral
4,-727.104370,62.355034,3.121181,15.064669,8.132434,1.927084,-3.274656,-3.761792,-9.750299,-4.853837,...,16.215825,44.607496,10.949800,5.019456,6.459340,4.461460,3.718469,4.304718,4.069478,calm
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
918,-520.086304,29.789667,-9.495820,5.796797,-5.974443,-14.387767,-14.708281,-10.981912,-8.302331,-1.795073,...,17.548885,44.429412,8.695367,8.006429,7.266029,6.345305,5.323645,5.854122,6.331995,happy
919,-531.830139,36.177265,-9.514291,5.053874,-3.236759,-11.967565,-10.899472,-9.773921,-6.809904,1.997820,...,18.580753,42.624524,8.868085,7.049018,6.032355,4.184395,4.555627,4.907770,7.809895,happy
920,-672.796814,44.611713,0.181294,11.717804,-3.714227,-4.868636,-11.445300,-8.688589,-2.259781,-4.132983,...,15.579213,43.473276,7.388364,6.343681,6.450891,4.783001,4.501908,3.705480,8.443843,sad
921,-667.400085,59.642807,3.031689,10.734396,-3.258640,-9.224376,-13.292101,-9.636474,-4.839901,-2.574258,...,16.325227,43.175374,8.811087,6.289404,6.330897,4.472781,4.525207,3.713926,8.547987,sad


In [9]:
df.sample(7)

,0,1,2,3,4,5,6,7,8,9,...,71,72,73,74,75,76,77,78,79,labels
637,-617.584351,55.408577,16.678368,11.436361,4.660233,4.480245,4.621804,2.243023,-3.979963,-2.853558,...,16.000226,45.394125,6.250928,5.151981,5.327503,3.810505,3.022207,3.116711,4.495709,fearful
709,-514.753357,50.858479,-20.525362,-3.608304,-9.386665,-9.762219,-8.020327,-20.655445,-9.601513,0.788539,...,19.215128,45.799315,6.986020,8.320150,7.797327,5.429456,6.217787,5.457580,5.095805,disgust
834,-666.552979,32.183926,-3.102011,4.109066,-8.511315,-7.834329,-10.507788,-10.454772,-3.351674,-7.774173,...,16.146836,40.529211,6.380233,6.505878,5.178259,4.150789,4.183855,3.698915,8.788203,surprised
636,-646.874451,56.470741,16.642214,14.731084,7.002588,6.844279,4.397758,2.498316,-2.653956,0.179521,...,16.136603,45.143346,6.044531,4.605108,5.130564,3.106213,2.995655,2.769865,4.787332,fearful
384,-595.376282,73.630943,-8.953110,14.371160,13.750571,-11.537727,-16.592735,-8.307069,-17.779594,-9.041086,...,18.899314,46.011502,7.151093,4.520040,4.978748,3.670724,3.363675,5.360175,3.912561,sad
583,-534.718079,48.854950,-8.730754,6.322406,-2.733296,-10.209534,-7.318885,-17.622160,-6.549259,-2.509206,...,19.133137,45.120729,4.498141,8.312349,6.562141,5.364825,4.474964,5.968540,5.226804,fearful
162,-390.576050,49.592781,-8.152850,7.991511,1.741827,5.736107,-9.831394,-15.284446,-9.608900,-12.327662,...,16.616964,44.560840,5.031928,5.782813,4.675770,3.235931,2.862025,2.579820,5.287937,fearful


In [10]:
df["labels"].value_counts()

labels
calm         128
happy        128
sad          123
angry        120
fearful      120
disgust      120
surprised    120
neutral       64
Name: count, dtype: int64

In [11]:
# Encode the labels (e.g., "happy", "angry", etc.)
lr = LabelEncoder()
df['labels'] = lr.fit_transform(df['labels'])

In [12]:
# Features and target variable
X = df.drop(columns='labels')
y = df['labels']

In [13]:
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [14]:
pt=PowerTransformer(method="yeo-johnson")
X_train = pt.fit_transform(X_train)
X_test = pt.transform(X_test)

In [15]:
from imblearn.over_sampling import RandomOverSampler
os = RandomOverSampler(sampling_strategy="auto", random_state=42)
X_train_resampled, y_train_resampled = os.fit_resample(X_train, y_train)

In [16]:
# Standardize the features
scaler = StandardScaler()
X_train_resampled = scaler.fit_transform(X_train_resampled)
X_test = scaler.transform(X_test)

In [17]:
from sklearn.svm import SVC
model=SVC(kernel="rbf",C=12,gamma=0.02)

In [18]:
# Train the model
model.fit(X_train_resampled, y_train_resampled)

# Predict on the test set
y_pred = model.predict(X_test)

# Calculate the accuracy
cr= classification_report(y_test, y_pred)
print("Classification_Report:", cr)

Classification_Report:               precision    recall  f1-score   support

           0       0.71      0.77      0.74        31
           1       0.73      0.94      0.82        32
           2       0.69      0.58      0.63        31
           3       0.54      0.68      0.60        22
           4       0.83      0.85      0.84        34
           5       0.62      0.33      0.43        15
           6       0.81      0.67      0.73        33
           7       0.78      0.76      0.77        33

    accuracy                           0.73       231
   macro avg       0.71      0.70      0.70       231
weighted avg       0.73      0.73      0.72       231



In [19]:
"""""CREMA-D -AUDIO-DATASET" # --- Tonnetz ---
        tonnetz = librosa.feature.tonnetz(y=librosa.effects.harmonic(y), sr=sr)
        tonnetz_mean = np.mean(tonnetz, axis=1)
        tonnetz_std = np.std(tonnetz, axis=1)

        # --- RMS Energy ---
        rms = librosa.feature.rms(y=y)
        rms_mean = np.mean(rms)
        rms_std = np.std(rms)

        # --- Zero Crossing Rate ---
        zcr = librosa.feature.zero_crossing_rate(y)
        zcr_mean = np.mean(zcr)
        zcr_std = np.std(zcr)

        # --- Spectral Contrast ---
        contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
        contrast_mean = np.mean(contrast, axis=1)
        contrast_std = np.std(contrast, axis=1)
        



        chroma_mean = np.mean(chroma, axis=1)
        chroma_std = np.std(chroma, axis=1)

        mfccs_mean = np.mean(mfccs, axis=1)
        mfccs_std = np.std(mfccs, axis=1)


# --- Zero Crossing Rate ---
        zcr = librosa.feature.zero_crossing_rate(y)
        zcr_mean = np.mean(zcr)
        zcr_std = np.std(zcr)

        # --- RMS Energy ---
        rms = librosa.feature.rms(y=y)
        rms_mean = np.mean(rms)
        rms_std = np.std(rms)

        # --- Tonnetz ---
        tonnetz = librosa.feature.tonnetz(y=librosa.effects.harmonic(y), sr=sr)
        tonnetz_mean = np.mean(tonnetz, axis=1)
        tonnetz_std = np.std(tonnetz, axis=1)

        # --- Spectral Contrast ---
        contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
        contrast_mean = np.mean(contrast, axis=1)
        contrast_std = np.std(contrast, axis=1)

        """
        

'""CREMA-D -AUDIO-DATASET" # --- Tonnetz ---\n        tonnetz = librosa.feature.tonnetz(y=librosa.effects.harmonic(y), sr=sr)\n        tonnetz_mean = np.mean(tonnetz, axis=1)\n        tonnetz_std = np.std(tonnetz, axis=1)\n\n        # --- RMS Energy ---\n        rms = librosa.feature.rms(y=y)\n        rms_mean = np.mean(rms)\n        rms_std = np.std(rms)\n\n        # --- Zero Crossing Rate ---\n        zcr = librosa.feature.zero_crossing_rate(y)\n        zcr_mean = np.mean(zcr)\n        zcr_std = np.std(zcr)\n\n        # --- Spectral Contrast ---\n        contrast = librosa.feature.spectral_contrast(y=y, sr=sr)\n        contrast_mean = np.mean(contrast, axis=1)\n        contrast_std = np.std(contrast, axis=1)\n        \n\n\n\n        chroma_mean = np.mean(chroma, axis=1)\n        chroma_std = np.std(chroma, axis=1)\n\n        mfccs_mean = np.mean(mfccs, axis=1)\n        mfccs_std = np.std(mfccs, axis=1)\n\n\n# --- Zero Crossing Rate ---\n        zcr = librosa.feature.zero_crossing_rat

In [20]:
import pickle

# Save the model, scaler, power transformer, and label encoder
with open("model.pkl", "wb") as f:
    pickle.dump(model, f)

with open("scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

with open("pt.pkl", "wb") as f:
    pickle.dump(pt, f)

with open("label_encoder.pkl", "wb") as f:
    pickle.dump(lr, f)
